In [ ]:
import os
import re
import pickle
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional

import base64
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils import weight_norm
from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler, RobustScaler

warnings.filterwarnings("ignore")

CLASS_NAMES   = ["Dispersed Flow", "Plug Flow", "Slug Flow"]
SUB_FOLDERS   = ["Dispersed-Flow", "Plug-Flow", "Slug-Flow"]   # label 0, 1, 2
VIDEO_EXTS    = {".mp4", ".avi", ".mov", ".mkv", ".wmv"}

DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# File-level inference (must match training in main.py)
# ---------------------------------------------------------------------------
# Files are no longer chopped into sliding windows. Every file's FULL raw
# pressure trace is resampled (linear interpolation) onto SERIES_LENGTH
# points before being fed to the TCN branch, and the hand-crafted features
# are extracted from the FULL raw trace (before resampling) -- exactly
# mirroring how every train/val/test sample was built in main.py.
SERIES_LENGTH = 90
MIN_TRACE_LEN = 8     # minimum raw samples needed to bother extracting features

# Architecture (must match training)
HIDDEN_SIZE     = 128
NUM_CLASSES     = 3
NUM_FEATURES    = 8
TCN_CHANNELS    = [32, 32, 64, 64, 64, 64, 64]
TCN_KERNEL_SIZE = 3


# ---------------------------------------------------------------------------
# TCN branch (identical to main.py -- weights are loaded from the same
# checkpoint, so the architecture must match exactly)
# ---------------------------------------------------------------------------
class Chomp1d(nn.Module):
    """Trims the extra right-side padding a causal dilated conv adds, so
    output length matches input length and no future timestep ever leaks
    into the receptive field of an earlier one."""

    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    """One residual block of two causal, dilated, weight-normalized 1-D
    convolutions -- the standard TCN building block (Bai et al., 2018)."""

    def __init__(
        self,
        n_inputs: int,
        n_outputs: int,
        kernel_size: int,
        dilation: int,
        dropout: float = 0.15,
    ):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.conv1 = weight_norm(nn.Conv1d(
            n_inputs, n_outputs, kernel_size, padding=padding, dilation=dilation
        ))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(nn.Conv1d(
            n_outputs, n_outputs, kernel_size, padding=padding, dilation=dilation
        ))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.relu1, self.dropout1,
            self.conv2, self.chomp2, self.relu2, self.dropout2,
        )
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self._init_weights()

    def _init_weights(self) -> None:
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


class TemporalConvNet(nn.Module):
    """Stack of TemporalBlocks with exponentially increasing dilation
    (1, 2, 4, 8, ...) so the receptive field grows to cover the whole
    file-level SERIES_LENGTH input."""

    def __init__(
        self,
        num_inputs: int,
        num_channels: list,
        kernel_size: int = 3,
        dropout: float = 0.15,
    ):
        super().__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation = 2 ** i
            in_ch = num_inputs if i == 0 else num_channels[i - 1]
            out_ch = num_channels[i]
            layers.append(TemporalBlock(
                in_ch, out_ch, kernel_size, dilation=dilation, dropout=dropout
            ))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


# ---------------------------------------------------------------------------
# Model -- file-level Multi-Task PINN (TCN branch, no separate physics_head;
# only classification + velocity heads, exactly matching main.py so the
# saved state-dict loads without any key mismatch).
# ---------------------------------------------------------------------------
class MultiTaskPINN(nn.Module):
    def __init__(
        self,
        window_size: int = SERIES_LENGTH,
        num_features: int = NUM_FEATURES,
        hidden_size: int = HIDDEN_SIZE,
        num_classes: int = NUM_CLASSES,
        cls_dropout: float = 0.4,
        reg_dropout: float = 0.15,
        tcn_channels: list = None,
        tcn_kernel_size: int = TCN_KERNEL_SIZE,
    ):
        super().__init__()

        tcn_channels = tcn_channels if tcn_channels is not None else TCN_CHANNELS

        self.tcn = TemporalConvNet(
            num_inputs=1,
            num_channels=tcn_channels,
            kernel_size=tcn_kernel_size,
            dropout=reg_dropout,
        )
        tcn_out_dim = tcn_channels[-1]

        self.fc_feats = nn.Sequential(
            nn.Linear(num_features, 64),
            nn.ReLU(),
            nn.Dropout(reg_dropout),
            nn.Linear(64, 64),
            nn.ReLU(),
        )

        self.shared = nn.Sequential(
            nn.Linear(tcn_out_dim + 64, hidden_size),
            nn.ReLU(),
            nn.Dropout(reg_dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(cls_dropout),
            nn.Linear(hidden_size // 2, num_classes),
        )

        self.velocity_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(reg_dropout),
            nn.Linear(hidden_size, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Linear(64, 2),   # [Vsg_scaled, Vsl_scaled]
        )

        self._init_weights()

    def _init_weights(self) -> None:
        for name, m in self.named_modules():
            if name.startswith("tcn."):
                continue
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, pressure_trace: torch.Tensor, features: torch.Tensor):
        tcn_out = self.tcn(pressure_trace.unsqueeze(1))   # (B, C, L)
        x = tcn_out[:, :, -1]                               # causal last-step summary
                                                              # of the WHOLE resampled file
        f = self.fc_feats(features)
        shared = self.shared(torch.cat([x, f], dim=1))
        return self.classifier(shared), self.velocity_head(shared)

@dataclass
class VideoEntry:
    """One record in the video index."""
    path:        str
    filename:    str
    regime_idx:  int
    regime_name: str
    vsg:         float
    vsl:         float


@dataclass
class RetrievalResult:
    """One Top-K hit returned to the caller."""
    rank:             int
    video_path:       str
    video_filename:   str
    regime_name:      str
    vsg:              float
    vsl:              float
    distance:         float          # Euclidean distance in (Vsg, Vsl) space
    query_vsg:        float
    query_vsl:        float
    query_regime:     str


class VideoRetrievalSystem:
    """
    Top-K video retrieval backed by the Paper-1 MultiTaskPINN -- FILE-LEVEL
    inference. Each query pressure trace is treated as one whole file:
    resampled once to SERIES_LENGTH for the TCN branch, with the 8
    hand-crafted features extracted once from the full raw trace. There is
    no sliding-window slicing and no aggregation across windows, since the
    model now consumes (and was trained on) one fixed-size vector per file.

    Parameters
    ----------
    video_base_dir  : Root directory containing Dispersed-Flow/, Plug-Flow/,
                      Slug-Flow/ sub-folders with video files.
    model_ckpt_path : Path to best_pinn_model.pth  (state-dict only).
    scalers_path    : Path to best_scalers.pkl  (dict with keys:
                      'pressure', 'features', 'vsg', 'vsl').
    cross_regime    : If True, Top-K search is performed across ALL regimes
                      (not just the predicted one).  Default: False.
    """

    def __init__(
        self,
        video_base_dir:  str,
        model_ckpt_path: str,
        scalers_path:    str,
        cross_regime:    bool = False,
    ):
        self.video_base_dir = Path(video_base_dir)
        self.cross_regime   = cross_regime

        print(f"[Init] Device : {DEVICE}")

        # 1. Load scalers
        print(f"[Init] Loading scalers  → {scalers_path}")
        with open(scalers_path, "rb") as fh:
            self.scalers: dict = pickle.load(fh)
        self._validate_scalers()

        # 2. Load model
        print(f"[Init] Loading model    → {model_ckpt_path}")
        self.model = MultiTaskPINN().to(DEVICE)
        state = torch.load(model_ckpt_path, map_location=DEVICE)
        # Handle both raw state-dict and checkpoint-dict formats
        if isinstance(state, dict) and "model_state_dict" in state:
            state = state["model_state_dict"]
        self.model.load_state_dict(state)
        self.model.eval()
        print(f"[Init] Model loaded successfully.")

        # 3. Build video index
        print(f"[Init] Indexing videos  → {self.video_base_dir}")
        self.index: List[VideoEntry] = self._build_index()

        # Pre-compute numpy arrays for fast distance calculation
        self._index_vsg = np.array([e.vsg for e in self.index], dtype=np.float32)
        self._index_vsl = np.array([e.vsl for e in self.index], dtype=np.float32)
        self._index_regime = np.array([e.regime_idx for e in self.index], dtype=np.int32)

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def retrieve(
        self,
        pressure_series: np.ndarray,
        k: int = 5,
    ) -> List[RetrievalResult]:
        """
        Run FILE-LEVEL inference on a raw pressure time-series (the whole
        file's trace, any length) and return the Top-K most similar videos
        ranked by Euclidean distance on (Vsg, Vsl).

        Parameters
        ----------
        pressure_series : 1-D float array of raw pressure values (barA)
                          for the ENTIRE file -- no minimum length beyond
                          MIN_TRACE_LEN samples, since the whole trace is
                          resampled to SERIES_LENGTH internally.
        k               : Number of results to return.

        Returns
        -------
        List[RetrievalResult] sorted by ascending distance (best match first).
        """
        pressure_series = np.asarray(pressure_series, dtype=np.float32)
        if len(pressure_series) < MIN_TRACE_LEN:
            raise ValueError(
                f"Pressure series length {len(pressure_series)} is shorter than "
                f"MIN_TRACE_LEN={MIN_TRACE_LEN}. Cannot run inference."
            )

        # ── Step 1: Inference (file-level, one forward pass) ───────────
        regime_idx, vsg_pred, vsl_pred, regime_probs = self._infer(pressure_series)
        regime_name = CLASS_NAMES[regime_idx]

        print(f"[Inference]  Regime  : {regime_name}  (confidence {regime_probs[regime_idx]:.1%})")
        print(f"[Inference]  Vsg     : {vsg_pred:.4f} m/s")
        print(f"[Inference]  Vsl     : {vsl_pred:.4f} m/s")

        # ── Step 2: Filter candidate pool ─────────────────────────────
        if self.cross_regime:
            candidate_mask = np.ones(len(self.index), dtype=bool)
        else:
            candidate_mask = self._index_regime == regime_idx

        candidate_indices = np.where(candidate_mask)[0]

        if len(candidate_indices) == 0:
            print(f"[Warning] No videos found for regime '{regime_name}'. "
                  f"Falling back to cross-regime search.")
            candidate_indices = np.arange(len(self.index))

        # ── Step 3: Euclidean distance on (Vsg, Vsl) ──────────────────
        cand_vsg = self._index_vsg[candidate_indices]
        cand_vsl = self._index_vsl[candidate_indices]

        distances = np.sqrt(
            (cand_vsg - vsg_pred) ** 2 +
            (cand_vsl - vsl_pred) ** 2
        )

        # ── Step 4: Top-K selection ────────────────────────────────────
        k_actual = min(k, len(candidate_indices))
        top_k_local = np.argsort(distances)[:k_actual]
        top_k_global = candidate_indices[top_k_local]

        results = []
        for rank, (global_idx, local_idx) in enumerate(
            zip(top_k_global, top_k_local), start=1
        ):
            entry = self.index[global_idx]
            results.append(RetrievalResult(
                rank           = rank,
                video_path     = entry.path,
                video_filename = entry.filename,
                regime_name    = entry.regime_name,
                vsg            = entry.vsg,
                vsl            = entry.vsl,
                distance       = float(distances[local_idx]),
                query_vsg      = vsg_pred,
                query_vsl      = vsl_pred,
                query_regime   = regime_name,
            ))

        return results

    def print_results(self, results: List[RetrievalResult]) -> None:
        """Pretty-print a retrieval result list to stdout."""
        if not results:
            print("[Results] No results to display.")
            return

        q = results[0]
        header = (
            f"\n{'='*70}\n"
            f"  Query  →  Regime: {q.query_regime}   "
            f"Vsg={q.query_vsg:.4f} m/s   Vsl={q.query_vsl:.4f} m/s\n"
            f"{'='*70}\n"
            f"  {'Rank':<5} {'Regime':<18} {'Vsg':>8} {'Vsl':>8} "
            f"{'Distance':>10}   Filename\n"
            f"  {'-'*65}"
        )
        print(header)
        for r in results:
            print(
                f"  {r.rank:<5} {r.regime_name:<18} "
                f"{r.vsg:>8.4f} {r.vsl:>8.4f} "
                f"{r.distance:>10.6f}   {r.video_filename}"
            )
        print(f"{'='*70}\n")

    def get_best_match(
        self,
        pressure_series: np.ndarray,
    ) -> Optional[RetrievalResult]:
        """Convenience wrapper — returns only the single best match."""
        results = self.retrieve(pressure_series, k=1)
        return results[0] if results else None

    def summary(self) -> None:
        """Print a summary of the video index."""
        print(f"\n{'='*50}")
        print(f"  VIDEO INDEX SUMMARY")
        print(f"{'='*50}")
        for idx, name in enumerate(CLASS_NAMES):
            count = sum(1 for e in self.index if e.regime_idx == idx)
            print(f"  {name:<20} : {count:>4} videos")
        print(f"  {'TOTAL':<20} : {len(self.index):>4} videos")
        print(f"{'='*50}\n")

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _validate_scalers(self) -> None:
        required = {"pressure", "features", "vsg", "vsl"}
        missing  = required - set(self.scalers.keys())
        if missing:
            raise KeyError(
                f"Scalers file is missing keys: {missing}. "
                f"Expected keys: {required}."
            )

    def _build_index(self) -> List[VideoEntry]:
        """
        Walk video_base_dir/SUB_FOLDERS and build a list of VideoEntry
        objects for every video file whose filename contains Vsg=X and Vsl=X.
        Files missing velocity info are skipped with a warning.
        """
        index = []
        skipped = 0

        for regime_idx, sub_folder in enumerate(SUB_FOLDERS):
            folder_path = self.video_base_dir / sub_folder
            if not folder_path.is_dir():
                print(f"  [Warning] Directory not found: {folder_path} — skipping.")
                continue

            for fname in sorted(os.listdir(folder_path)):
                ext = Path(fname).suffix.lower()
                if ext not in VIDEO_EXTS:
                    continue

                vsg, vsl = extract_velocities_from_filename(fname)
                if vsg is None or vsl is None:
                    print(f"  [Warning] Cannot parse Vsg/Vsl from '{fname}' — skipping.")
                    skipped += 1
                    continue

                index.append(VideoEntry(
                    path        = str(folder_path / fname),
                    filename    = fname,
                    regime_idx  = regime_idx,
                    regime_name = CLASS_NAMES[regime_idx],
                    vsg         = vsg,
                    vsl         = vsl,
                ))

        if skipped:
            print(f"  [Warning] Skipped {skipped} file(s) with unparseable filenames.")

        return index

    def _infer(self, pressure_series: np.ndarray):
        """
        FILE-LEVEL inference: resample the ENTIRE pressure trace to
        SERIES_LENGTH for the TCN branch, extract the 8 hand-crafted
        features from the FULL raw trace, and run a single forward pass
        -- exactly mirroring `run_inference` in main.py. No sliding
        window, no per-window aggregation.

        Returns:
            regime_idx  : int
            vsg_pred    : float  (original scale, m/s)
            vsl_pred    : float  (original scale, m/s)
            regime_probs: np.ndarray  shape (NUM_CLASSES,)
        """
        pressure_series = np.asarray(pressure_series, dtype=np.float32)

        feats = extract_pressure_features(pressure_series)
        pressure_fixed = resample_series(pressure_series, SERIES_LENGTH)

        p_scaled = self.scalers["pressure"].transform(pressure_fixed.reshape(1, -1))
        f_scaled = self.scalers["features"].transform(feats.reshape(1, -1))

        p_tensor = torch.tensor(p_scaled, dtype=torch.float32).to(DEVICE)
        f_tensor = torch.tensor(f_scaled, dtype=torch.float32).to(DEVICE)

        with torch.no_grad():
            cls_logits, vel_pred = self.model(p_tensor, f_tensor)

        logits_arr = cls_logits.cpu().numpy()          # (1, 3)
        softmax = np.exp(logits_arr) / np.exp(logits_arr).sum(axis=1, keepdims=True)
        regime_probs = softmax[0]                       # (3,)
        regime_idx = int(np.argmax(regime_probs))

        vsg_scaled = vel_pred[:, 0:1].cpu().numpy()      # (1, 1)
        vsl_scaled = vel_pred[:, 1:2].cpu().numpy()      # (1, 1)

        vsg_pred = float(self.scalers["vsg"].inverse_transform(vsg_scaled)[0, 0])
        vsl_pred = float(self.scalers["vsl"].inverse_transform(vsl_scaled)[0, 0])

        return regime_idx, vsg_pred, vsl_pred, regime_probs

def _video_to_b64(path: str) -> str:
    """Read a video file from disk and return a base64-encoded string."""
    with open(path, "rb") as fh:
        return base64.b64encode(fh.read()).decode("utf-8")


def _mime_type(path: str) -> str:
    ext = Path(path).suffix.lower()
    return {
        ".mp4":  "video/mp4",
        ".avi":  "video/x-msvideo",
        ".mov":  "video/quicktime",
        ".mkv":  "video/x-matroska",
        ".wmv":  "video/x-ms-wmv",
        ".webm": "video/webm",
    }.get(ext, "video/mp4")


def _video_html(path: str, width: int = 640) -> str:
    """Return an HTML5 <video> tag with the file embedded as base64."""
    b64   = _video_to_b64(path)
    mime  = _mime_type(path)
    return f"""
    <video width="{width}" controls autoplay loop
           style="border-radius:8px; border:2px solid #4a90d9; margin-top:6px;">
        <source src="data:{mime};base64,{b64}" type="{mime}">
        Your browser does not support the HTML5 video tag.
    </video>
    """


def _badge(text: str, color: str = "#4a90d9") -> str:
    return (
        f'<span style="background:{color};color:#fff;padding:2px 8px;'
        f'border-radius:4px;font-size:12px;font-weight:600;">{text}</span>'
    )


REGIME_COLORS = {
    "Dispersed Flow": "#2ecc71",
    "Plug Flow":      "#e67e22",
    "Slug Flow":      "#9b59b6",
}

class RetrievalPlayer:
    """
    Interactive Jupyter widget that lets you browse Top-K retrieval results
    and play each matched video inline.

    Parameters
    ----------
    results  : List of RetrievalResult objects from VideoRetrievalSystem.retrieve()
    width    : Video display width in pixels (default 640).
    autoplay : Whether the video starts playing automatically (default True).

    Example
    -------
    results = vrs.retrieve(pressure_series, k=5)
    player  = RetrievalPlayer(results)
    player.show()
    """

    def __init__(self, results, width: int = 640, autoplay: bool = True):
        if not results:
            raise ValueError("results list is empty — nothing to play.")

        self.results  = results
        self.width    = width
        self.autoplay = autoplay
        self._current = 0          # currently displayed result index

        self._build_ui()

    # ------------------------------------------------------------------
    # Public
    # ------------------------------------------------------------------

    def show(self):
        """Render the player widget in the current Jupyter cell."""
        display(self._root)
        self._render(0)

    # ------------------------------------------------------------------
    # UI construction
    # ------------------------------------------------------------------

    def _build_ui(self):
        n = len(self.results)

        # ── Header ────────────────────────────────────────────────────
        q = self.results[0]
        header_html = (
            f'<div style="font-family:monospace;padding:8px 0;">'
            f'<b>Query</b> &nbsp;→&nbsp; '
            f'{_badge(q.query_regime, REGIME_COLORS.get(q.query_regime,"#555"))} &nbsp; '
            f'Vsg&nbsp;=&nbsp;<b>{q.query_vsg:.4f}</b>&nbsp;m/s &nbsp; '
            f'Vsl&nbsp;=&nbsp;<b>{q.query_vsl:.4f}</b>&nbsp;m/s'
            f'</div>'
        )
        self._header = widgets.HTML(value=header_html)

        # ── Navigation buttons ─────────────────────────────────────────
        btn_style = dict(button_style="", layout=widgets.Layout(width="110px"))

        self._btn_prev = widgets.Button(description="◀  Prev", **btn_style)
        self._btn_next = widgets.Button(description="Next  ▶", **btn_style)
        self._rank_label = widgets.HTML()

        self._btn_prev.on_click(lambda _: self._navigate(-1))
        self._btn_next.on_click(lambda _: self._navigate(+1))

        nav_bar = widgets.HBox(
            [self._btn_prev, self._rank_label, self._btn_next],
            layout=widgets.Layout(align_items="center", gap="12px", margin="4px 0"),
        )

        # ── Result metadata panel ──────────────────────────────────────
        self._meta_panel = widgets.HTML()

        # ── Video output area ──────────────────────────────────────────
        self._video_out = widgets.Output(
            layout=widgets.Layout(margin="6px 0")
        )

        # ── Results table (collapsed by default) ──────────────────────
        self._table_out  = widgets.Output()
        self._table_acc  = widgets.Accordion(children=[self._table_out])
        self._table_acc.set_title(0, f"📋  All {n} results")
        self._table_acc.selected_index = None   # collapsed

        self._render_table()

        # ── Root layout ────────────────────────────────────────────────
        self._root = widgets.VBox(
            [
                self._header,
                widgets.HTML("<hr style='margin:4px 0;border-color:#ddd;'>"),
                nav_bar,
                self._meta_panel,
                self._video_out,
                self._table_acc,
            ],
            layout=widgets.Layout(
                padding="10px",
                border="1px solid #ddd",
                border_radius="8px",
                max_width="700px",
            ),
        )

    # ------------------------------------------------------------------
    # Rendering helpers
    # ------------------------------------------------------------------

    def _navigate(self, delta: int):
        new_idx = (self._current + delta) % len(self.results)
        self._render(new_idx)

    def _render(self, idx: int):
        self._current = idx
        r = self.results[idx]
        n = len(self.results)

        # Rank label
        self._rank_label.value = (
            f'<span style="font-size:14px;font-weight:600;">'
            f'Result {idx+1} / {n}</span>'
        )

        # Prev / Next button states
        self._btn_prev.disabled = (n == 1)
        self._btn_next.disabled = (n == 1)

        # Metadata panel
        regime_color = REGIME_COLORS.get(r.regime_name, "#555")
        self._meta_panel.value = f"""
        <div style="font-family:monospace;font-size:13px;
                    background:#f8f8f8;border-radius:6px;padding:8px 12px;
                    line-height:1.8;">
            <b>Rank</b>         &nbsp;→&nbsp; {r.rank} &nbsp;
            {_badge(r.regime_name, regime_color)}<br>
            <b>Vsg</b>          &nbsp;→&nbsp; {r.vsg:.4f} m/s
            &nbsp;&nbsp;
            <b>Vsl</b>          &nbsp;→&nbsp; {r.vsl:.4f} m/s<br>
            <b>Distance</b>     &nbsp;→&nbsp; {r.distance:.6f}<br>
            <b>File</b>         &nbsp;→&nbsp;
            <span style="color:#555;">{r.video_filename}</span>
        </div>
        """

        # Video
        self._video_out.clear_output(wait=True)
        with self._video_out:
            if not os.path.isfile(r.video_path):
                display(HTML(
                    f'<p style="color:red;font-family:monospace;">'
                    f'⚠ File not found:<br>{r.video_path}</p>'
                ))
            else:
                try:
                    display(HTML(_video_html(r.video_path, width=self.width)))
                except Exception as exc:
                    display(HTML(
                        f'<p style="color:red;font-family:monospace;">'
                        f'⚠ Could not load video: {exc}</p>'
                    ))

    def _render_table(self):
        """Render the full results table inside the accordion."""
        rows = ""
        for r in self.results:
            color  = REGIME_COLORS.get(r.regime_name, "#555")
            is_cur = r.rank == 1
            bg     = "#fffbe6" if is_cur else "#fff"
            rows += (
                f'<tr style="background:{bg};">'
                f'<td style="text-align:center;">{r.rank}</td>'
                f'<td><span style="background:{color};color:#fff;'
                f'padding:1px 6px;border-radius:3px;font-size:11px;">'
                f'{r.regime_name}</span></td>'
                f'<td style="text-align:right;">{r.vsg:.4f}</td>'
                f'<td style="text-align:right;">{r.vsl:.4f}</td>'
                f'<td style="text-align:right;">{r.distance:.6f}</td>'
                f'<td style="font-size:11px;color:#555;">{r.video_filename}</td>'
                f'</tr>'
            )

        table_html = f"""
        <div style="overflow-x:auto;">
        <table style="border-collapse:collapse;width:100%;
                      font-family:monospace;font-size:12px;">
            <thead>
                <tr style="background:#4a90d9;color:#fff;">
                    <th style="padding:5px 8px;">Rank</th>
                    <th style="padding:5px 8px;">Regime</th>
                    <th style="padding:5px 8px;">Vsg (m/s)</th>
                    <th style="padding:5px 8px;">Vsl (m/s)</th>
                    <th style="padding:5px 8px;">Distance</th>
                    <th style="padding:5px 8px;">Filename</th>
                </tr>
            </thead>
            <tbody>{rows}</tbody>
        </table>
        </div>
        """
        with self._table_out:
            display(HTML(table_html))


if __name__ == "__main__":
    
    VIDEO_BASE_DIR  = r"./Videos"
    MODEL_CKPT_PATH = r"models\best_pinn_model.pth"
    SCALERS_PATH    = r"models\best_scalers.pkl"

    vrs = VideoRetrievalSystem(
        video_base_dir  = VIDEO_BASE_DIR,
        model_ckpt_path = MODEL_CKPT_PATH,
        scalers_path    = SCALERS_PATH,
    )

    print("[Demo] Running Top-5 retrieval on a pressure signal ...\n")
    demo_pressure = np.random.randn(200).astype(np.float32) * 0.01 + 1.5

    results = vrs.retrieve(pressure_series=demo_pressure, k=5)
    vrs.print_results(results)

    best = results[0]
    print(f"Best match path   : {best.video_path}")
    print(f"Best match regime : {best.regime_name}")
    print(f"Best match Vsg    : {best.vsg:.4f} m/s")
    print(f"Best match Vsl    : {best.vsl:.4f} m/s")
    print(f"Distance          : {best.distance:.6f}")

    player = RetrievalPlayer(results, width=640)
    player.show()

[Init] Device : cuda
[Init] Loading scalers  → models\best_scalers.pkl
[Init] Loading model    → models\best_pinn_model.pth
[Init] Model loaded successfully.
[Init] Indexing videos  → Videos
[Demo] Running Top-5 retrieval on a pressure signal ...

[Inference]  Regime  : Dispersed Flow  (confidence 99.7%)
[Inference]  Vsg     : 0.1747 m/s
[Inference]  Vsl     : 2.5047 m/s

  Query  →  Regime: Dispersed Flow   Vsg=0.1747 m/s   Vsl=2.5047 m/s
  Rank  Regime                  Vsg      Vsl   Distance   Filename
  -----------------------------------------------------------------
  1     Dispersed Flow       0.2100   2.6000   0.101595   df-Vsg=0.21-Vsl=2.6-vid8.mp4
  2     Dispersed Flow       0.2630   2.6000   0.129874   df-Vsg=0.263-Vsl=2.6-vid13.mp4
  3     Dispersed Flow       0.0530   2.6000   0.154602   df-Vsg=0.053-Vsl=2.6-vid10.mp4
  4     Dispersed Flow       0.1640   2.2900   0.214986   df-Vsg=0.164-Vsl=2.29-vid12.mp4
  5     Dispersed Flow       0.1100   2.2900   0.224267   df-Vsg=0